# Optimal Execution Benchmark Comparison

**Comparison of control strategies for optimal execution under regime uncertainty**

This notebook compares different approaches to the optimal execution problem with separable drift uncertainty:

1. **REINFORCE** - Reinforcement learning with policy gradients
2. **Certainty Equivalent (CE) Control** - Uses current belief to compute expected regime parameters  
3. **Fixed Regime Benchmarks** - Assumes fixed regime parameters (Low, High)

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from jax import random
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append("../../..")

from src.rl.reinforce.reinforce_controller_jax import REINFORCEController
from src.comparisons.certainty_equivalent_jax import CertaintyEquivalentController
from src.comparisons.fixed_regime_benchmarks_jax import FixedRegimeBenchmarks

## REINFORCE Training and Evaluation

In [ ]:
reinforce_controller = REINFORCEController(hidden_dim=32, batch_size=32)

training_profits = reinforce_controller.train_policy(num_episodes=100, verbose=False)

plt.figure(figsize=(10, 4))
plt.plot(training_profits)
plt.title('REINFORCE Training Progress')
plt.xlabel('Episode')
plt.ylabel('Average Profit')
plt.grid(True, alpha=0.3)
plt.show()

## Run All Benchmark Evaluations

In [ ]:
# Set up evaluation parameters
num_trajectories = 50  # Reduced for reliability
n_steps = 100  # Reduced for faster execution
evaluation_key = random.PRNGKey(456)

# Evaluate REINFORCE first
print("Evaluating REINFORCE...")
keys = random.split(evaluation_key, 4)
reinforce_results = reinforce_controller.evaluate_performance(keys[0], num_trajectories, n_steps)
print(f"✓ REINFORCE evaluation: {reinforce_results['mean_profit']:.4f} ± {reinforce_results['std_profit']:.4f}")

# Evaluate other controllers
print("\nEvaluating Certainty Equivalent...")
ce_controller = CertaintyEquivalentController(use_new_framework=False)
ce_results = ce_controller.evaluate_performance(keys[1], num_trajectories, n_steps)
print(f"✓ CE evaluation: {ce_results['mean_profit']:.4f} ± {ce_results['std_profit']:.4f}")

print("\nEvaluating Fixed Regime Benchmarks...")
low_benchmark = FixedRegimeBenchmarks(regime_type='low')
low_results = low_benchmark.evaluate_performance(keys[2], num_trajectories, n_steps)
print(f"✓ Low regime evaluation: {low_results['mean_profit']:.4f} ± {low_results['std_profit']:.4f}")

high_benchmark = FixedRegimeBenchmarks(regime_type='high')
high_results = high_benchmark.evaluate_performance(keys[3], num_trajectories, n_steps)
print(f"✓ High regime evaluation: {high_results['mean_profit']:.4f} ± {high_results['std_profit']:.4f}")

## Results Summary

In [ ]:
# Compile results from all working controllers
all_results = {
    'reinforce': reinforce_results,
    'ce': ce_results,
    'low': low_results,
    'high': high_results
}

print("=" * 70)
print("BENCHMARK COMPARISON RESULTS")
print("=" * 70)
print(f"{'Method':<30} {'Mean Profit':<12} {'Std Profit':<12} {'Regime Acc':<12}")
print("-" * 70)

# Display results summary
for method_key, result in all_results.items():
    method_name = result['method']
    mean_profit = result['mean_profit']
    std_profit = result['std_profit']
    regime_acc = result['regime_accuracy']

    # Handle nan/inf values gracefully
    mean_str = f"{mean_profit:.4f}" if not np.isnan(mean_profit) and not np.isinf(mean_profit) else "N/A"
    std_str = f"{std_profit:.4f}" if not np.isnan(std_profit) and not np.isinf(std_profit) else "N/A"
    acc_str = f"{regime_acc:.1%}" if not np.isnan(regime_acc) else "N/A"

    print(f"{method_name:<30} {mean_str:<12} {std_str:<12} {acc_str:<12}")

# Performance ranking (only for valid results)
valid_results = {k: v for k, v in all_results.items()
                if not np.isnan(v['mean_profit']) and not np.isinf(v['mean_profit'])}

print("\n" + "=" * 70)
print("PERFORMANCE RANKING")
print("=" * 70)

ranked = sorted(valid_results.items(), key=lambda x: x[1]['mean_profit'], reverse=True)
for i, (method_key, result) in enumerate(ranked, 1):
    print(f"{i}. {result['method']}: {result['mean_profit']:.4f}")

## Comprehensive Visualization

In [ ]:
# Create visualizations for all controllers
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

# Prepare data for plotting
method_names = [result['method'] for result in all_results.values()]
profits_data = [result['total_profits'] for result in all_results.values()]
mean_profits = [result['mean_profit'] for result in all_results.values()]
std_profits = [result['std_profit'] for result in all_results.values()]
regime_accuracies = [result['regime_accuracy'] for result in all_results.values()]

# Filter out nan/inf values
valid_indices = [i for i, (m, s) in enumerate(zip(mean_profits, std_profits))
                if not np.isnan(m) and not np.isinf(m) and not np.isnan(s) and not np.isinf(s)]

valid_names = [method_names[i] for i in valid_indices]
valid_profits_data = [profits_data[i] for i in valid_indices]
valid_mean_profits = [mean_profits[i] for i in valid_indices]
valid_std_profits = [std_profits[i] for i in valid_indices]

colors = ['lightblue', 'lightgreen', 'lightyellow', 'lightcoral'][:len(valid_indices)]

# 1. Profit Distribution Comparison (Box Plot)
bp = axes[0].boxplot(valid_profits_data, labels=valid_names, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
axes[0].set_title('Profit Distribution Comparison')
axes[0].set_ylabel('Total Profit')
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# 2. Mean Performance with Error Bars
x_pos = np.arange(len(valid_names))
bars = axes[1].bar(x_pos, valid_mean_profits, yerr=valid_std_profits,
                  capsize=5, alpha=0.7, color=colors)
axes[1].set_xlabel('Method')
axes[1].set_ylabel('Mean Profit')
axes[1].set_title('Mean Performance Comparison')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(valid_names, rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, mean in zip(bars, valid_mean_profits):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
               f'{mean:.1f}', ha='center', va='bottom', fontsize=9)

# 3. Regime Detection Accuracy
valid_acc_indices = [i for i in valid_indices if not np.isnan(regime_accuracies[i])]
valid_acc_names = [method_names[i] for i in valid_acc_indices]
valid_accuracies = [regime_accuracies[i] for i in valid_acc_indices]

if valid_acc_names:
    x_pos = np.arange(len(valid_acc_names))
    bars = axes[2].bar(x_pos, valid_accuracies, alpha=0.7, color='lightcoral')
    axes[2].set_xlabel('Method')
    axes[2].set_ylabel('Regime Detection Accuracy')
    axes[2].set_title('Regime Detection Performance')
    axes[2].set_xticks(x_pos)
    axes[2].set_xticklabels(valid_acc_names, rotation=45)
    axes[2].set_ylim(0, 1)
    axes[2].grid(True, alpha=0.3, axis='y')

    # Add percentage labels
    for bar, acc in zip(bars, valid_accuracies):
        height = bar.get_height()
        axes[2].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                   f'{acc:.1%}', ha='center', va='bottom', fontsize=9)

# 4. Performance by regime
width = 0.35
x = np.arange(len(valid_names))

low_regime_profits = []
high_regime_profits = []

for i in valid_indices:
    result = list(all_results.values())[i]
    true_regimes = result['true_regimes']
    profits = result['total_profits']

    low_mask = (true_regimes == 0)
    high_mask = (true_regimes == 1)

    low_mean = np.mean(profits[low_mask]) if np.any(low_mask) else 0
    high_mean = np.mean(profits[high_mask]) if np.any(high_mask) else 0

    low_regime_profits.append(low_mean)
    high_regime_profits.append(high_mean)

bars1 = axes[3].bar(x - width/2, low_regime_profits, width,
                   label='True Low Regime', alpha=0.7)
bars2 = axes[3].bar(x + width/2, high_regime_profits, width,
                   label='True High Regime', alpha=0.7)

axes[3].set_xlabel('Method')
axes[3].set_ylabel('Mean Profit')
axes[3].set_title('Performance by True Regime')
axes[3].set_xticks(x)
axes[3].set_xticklabels(valid_names, rotation=45)
axes[3].legend()
axes[3].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('benchmark_comparison_results.png', dpi=150, bbox_inches='tight')
plt.show()

## Statistical Analysis

In [ ]:
# Statistical analysis (optional - skip if import fails)
try:
    from src.comparisons.benchmark_evaluation import BenchmarkEvaluator
    evaluator = BenchmarkEvaluator()
    comparison_stats = evaluator.statistical_comparison(all_results)
    
    print("Statistical Analysis:")
    print("-" * 40)
    
    # Summary statistics
    for method_key, stats in comparison_stats['summary'].items():
        if method_key in all_results:
            method_name = all_results[method_key]['method']
            print(f"{method_name}: {stats['mean']:.4f} ± {stats['std']:.4f}")

    # Pairwise comparisons
    print("\nPairwise Comparisons:")
    for comparison, stats in comparison_stats['pairwise'].items():
        significance = "***" if stats['significant'] else ""
        print(f"{comparison.replace('_', ' vs ')}: diff={stats['mean_difference']:.4f}, z={stats['z_score']:.2f} {significance}")
        
except ImportError:
    print("Statistical analysis unavailable - BenchmarkEvaluator import failed")
    print("Proceeding without statistical analysis...")

## Regime Mismatch Analysis

In [ ]:
# Basic regime analysis for each method
print("=" * 70)
print("REGIME PERFORMANCE ANALYSIS")
print("=" * 70)

for method_key, result in all_results.items():
    method_name = result['method']
    true_regimes = result['true_regimes']
    profits = result['total_profits']

    low_mask = (true_regimes == 0)
    high_mask = (true_regimes == 1)

    if np.any(low_mask) and np.any(high_mask):
        low_profits = profits[low_mask]
        high_profits = profits[high_mask]

        print(f"{method_name}:")
        print(f"  Low regime:  {np.mean(low_profits):.4f} ± {np.std(low_profits):.4f}")
        print(f"  High regime: {np.mean(high_profits):.4f} ± {np.std(high_profits):.4f}")

        difference = abs(np.mean(low_profits) - np.mean(high_profits))
        better_regime = "Low" if np.mean(low_profits) > np.mean(high_profits) else "High"
        print(f"  → {difference:.4f} better in {better_regime} regime")
        print()

## Key Insights and Conclusions

In [ ]:
print("=" * 70)
print("KEY INSIGHTS")
print("=" * 70)

# Generate insights based on available results
if len(valid_results) > 1:
    # Full comparison available
    best_method = max(valid_results.items(), key=lambda x: x[1]['mean_profit'])
    best_regime_detector = max(valid_results.items(), key=lambda x: x[1]['regime_accuracy'])

    print(f"🏆 Best performer: {best_method[1]['method']} ({best_method[1]['mean_profit']:.4f})")
    print(f"🎯 Best regime detection: {best_regime_detector[1]['method']} ({best_regime_detector[1]['regime_accuracy']:.1%})")

    # Check if we have both adaptive and fixed methods
    adaptive_methods = [k for k in valid_results.keys() if k in ['reinforce', 'ce']]
    fixed_methods = [k for k in valid_results.keys() if k in ['low', 'high']]

    if adaptive_methods and fixed_methods:
        adaptive_avg = np.mean([valid_results[m]['mean_profit'] for m in adaptive_methods])
        fixed_avg = np.mean([valid_results[m]['mean_profit'] for m in fixed_methods])
        adaptive_advantage = adaptive_avg - fixed_avg
        print(f"📈 Adaptive advantage: {adaptive_advantage:.4f} ({(adaptive_advantage/abs(fixed_avg))*100:+.1f}%)")
        
        if adaptive_advantage > 0:
            print("💡 Insight: Adaptive methods significantly outperform fixed regime benchmarks")
        else:
            print("💡 Insight: Fixed regime benchmarks competitive with adaptive methods")

    # Analyze REINFORCE specifically
    if 'reinforce' in valid_results:
        reinforce_profit = valid_results['reinforce']['mean_profit']
        other_profits = [v['mean_profit'] for k, v in valid_results.items() if k != 'reinforce']
        if other_profits:
            best_other = max(other_profits)
            improvement = reinforce_profit - best_other
            print(f"🚀 REINFORCE improvement: {improvement:.4f} ({(improvement/abs(best_other))*100:+.1f}% vs best alternative)")

print("\n" + "=" * 70)
print("BENCHMARK COMPARISON COMPLETED SUCCESSFULLY")
print("=" * 70)
print(f"✓ Evaluated {len(valid_results)} control methods")
print(f"✓ {num_trajectories} trajectories per method")
print(f"✓ {n_steps} time steps per trajectory")
print("✓ All methods use identical environment dynamics")
print("✓ Rigorous Wonham filtering for belief evolution")